
# Phase 03 — Data Cleaning and Validation

## Objective

Create a reproducible, validated, user-level analysis dataset for the A/B Test and Experiment Analysis project.

## Source Dataset

- Original file: `data/ab_data.csv`
- Original records: 294,478
- Unique users: 290,584
- Analysis unit: Individual user

## Data Quality Issues Identified

- Repeated users: 3,894
- Users appearing in both experiment groups: 1,895
- Users repeated within one group: 1,999
- Group–landing page mismatches: 3,893 records

## Statistical Safeguards

1. Preserve the original dataset unchanged.
2. Establish and document cleaning decisions before implementing exclusions.
3. Preserve the distinction between assigned group and observed landing page.
4. Avoid outcome-based selection of cleaning rules.
5. Ensure one independent observation per eligible user.
6. Validate retained and excluded records.
7. Perform SRM checks using the approved threshold of p < 0.001.
8. Do not conduct treatment-effect hypothesis testing during Phase 03.

## Current Status

Notebook initialized. Cleaning and eligibility policies are not yet finalized.


In [2]:

# Step 03.14 — Load Original Dataset Safely

from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project root from the notebooks directory.
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Define the original dataset path.
raw_data_path = project_root / "data" / "ab_data.csv"

# Verify that the source file exists.
assert raw_data_path.is_file(), (
    f"Dataset not found: {raw_data_path}"
)

# Load the original dataset without modifying it.
df_raw = pd.read_csv(raw_data_path)

# Verify the expected original structure.
expected_columns = [
    "user_id",
    "timestamp",
    "group",
    "landing_page",
    "converted",
]

assert df_raw.shape == (294478, 5)
assert df_raw.columns.tolist() == expected_columns

print("========== PHASE 03: RAW DATA LOAD ==========")
print(f"Dataset path: {raw_data_path}")
print(f"Total records: {len(df_raw):,}")
print(f"Total columns: {df_raw.shape[1]}")
print(f"Unique users: {df_raw['user_id'].nunique():,}")
print("Original dataset loaded successfully.")


========== PHASE 03: RAW DATA LOAD ==========
Dataset path: c:\Users\kunal\OneDrive\Desktop\ab-test-experiment-analysis\data\ab_data.csv
Total records: 294,478
Total columns: 5
Unique users: 290,584
Original dataset loaded successfully.


In [3]:

# Step 03.15 — Create a Safe Working Copy

# Preserve the original DataFrame.
df = df_raw.copy(deep=True)

# Verify the working copy matches the original.
assert df.shape == df_raw.shape
assert df.equals(df_raw)

# Verify the two DataFrames are separate objects.
assert df is not df_raw

print("========== WORKING COPY VALIDATION ==========")
print(f"Original records: {len(df_raw):,}")
print(f"Working records: {len(df):,}")
print(f"Original columns: {df_raw.shape[1]}")
print(f"Working columns: {df.shape[1]}")
print(f"Data identical: {df.equals(df_raw)}")
print(f"Separate DataFrames: {df is not df_raw}")
print("Safe working copy created successfully.")


========== WORKING COPY VALIDATION ==========
Original records: 294,478
Working records: 294,478
Original columns: 5
Working columns: 5
Data identical: True
Separate DataFrames: True
Safe working copy created successfully.


In [4]:

# Step 03.16 — Validate Column Names and Data Types

expected_columns = [
    "user_id",
    "timestamp",
    "group",
    "landing_page",
    "converted"
]

print("========== COLUMN VALIDATION ==========")

print("Actual columns:", df.columns.tolist())
print("Expected columns:", expected_columns)

assert df.columns.tolist() == expected_columns

print("\n========== DATA TYPES ==========")
print(df.dtypes)

print("\n========== STRUCTURE CHECK ==========")
print("Column count:", df.shape[1])
print("Column names valid:", df.columns.tolist() == expected_columns)
print("Dataset structure validated successfully.")


========== COLUMN VALIDATION ==========
Actual columns: ['user_id', 'timestamp', 'group', 'landing_page', 'converted']
Expected columns: ['user_id', 'timestamp', 'group', 'landing_page', 'converted']

========== DATA TYPES ==========
user_id         int64
timestamp         str
group             str
landing_page      str
converted       int64
dtype: object

========== STRUCTURE CHECK ==========
Column count: 5
Column names valid: True
Dataset structure validated successfully.


In [5]:

# Step 03.17 — Validate Timestamp Conversion

import pandas as pd

# Convert timestamps into a temporary Series.
# Do not modify df or df_raw yet.
parsed_timestamps = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

invalid_timestamps = parsed_timestamps.isna().sum()

print("========== TIMESTAMP VALIDATION ==========")
print(f"Total records: {len(df):,}")
print(f"Valid timestamps: {parsed_timestamps.notna().sum():,}")
print(f"Invalid timestamps: {invalid_timestamps:,}")
print(f"Parsed data type: {parsed_timestamps.dtype}")

print("\n========== TIMESTAMP RANGE ==========")
print("Earliest timestamp:", parsed_timestamps.min())
print("Latest timestamp:", parsed_timestamps.max())

print("\nOriginal timestamp column unchanged:",
      df["timestamp"].dtype == "object")

assert invalid_timestamps == 0

print("\nTimestamp validation successful.")


========== TIMESTAMP VALIDATION ==========
Total records: 294,478
Valid timestamps: 294,478
Invalid timestamps: 0
Parsed data type: datetime64[us]

========== TIMESTAMP RANGE ==========
Earliest timestamp: 2017-01-02 13:42:05.378582
Latest timestamp: 2017-01-24 13:41:54.460509

Original timestamp column unchanged: False

Timestamp validation successful.


In [6]:

# Step 03.18 — Verify Original Timestamp Preservation

print("========== ORIGINAL DATA PRESERVATION ==========")

print("Raw timestamp dtype:", df_raw["timestamp"].dtype)
print("Working timestamp dtype:", df["timestamp"].dtype)

timestamps_identical = df["timestamp"].equals(
    df_raw["timestamp"]
)

print("Timestamp values identical:", timestamps_identical)

raw_data_preserved = df.equals(df_raw)

print("Entire original dataset preserved:", raw_data_preserved)

assert timestamps_identical
assert raw_data_preserved

print("\nOriginal data preservation verified successfully.")


========== ORIGINAL DATA PRESERVATION ==========
Raw timestamp dtype: str
Working timestamp dtype: str
Timestamp values identical: True
Entire original dataset preserved: True

Original data preservation verified successfully.


In [7]:

# Step 03.19 — Validate Experiment Group Labels

expected_groups = {"control", "treatment"}

print("========== EXPERIMENT GROUP VALIDATION ==========")

# Inspect all observed group labels.
group_counts = df["group"].value_counts(dropna=False)

print("\nObserved group distribution:")
print(group_counts)

# Identify unexpected labels.
observed_groups = set(df["group"].dropna().unique())

unexpected_groups = observed_groups - expected_groups

missing_group_values = df["group"].isna().sum()

print("\n========== VALIDATION RESULTS ==========")
print("Expected groups:", expected_groups)
print("Observed groups:", observed_groups)
print("Unexpected groups:", unexpected_groups)
print("Missing group values:", missing_group_values)

assert observed_groups == expected_groups
assert missing_group_values == 0

print("\nExperiment group labels validated successfully.")


========== EXPERIMENT GROUP VALIDATION ==========

Observed group distribution:
group
treatment    147276
control      147202
Name: count, dtype: int64

========== VALIDATION RESULTS ==========
Expected groups: {'treatment', 'control'}
Observed groups: {'treatment', 'control'}
Unexpected groups: set()
Missing group values: 0

Experiment group labels validated successfully.


In [8]:

# Step 03.20 — Validate Landing Page Labels

expected_pages = {"old_page", "new_page"}

print("========== LANDING PAGE VALIDATION ==========")

# Inspect landing page distribution.
page_counts = df["landing_page"].value_counts(dropna=False)

print("\nObserved landing page distribution:")
print(page_counts)

# Identify unexpected landing page labels.
observed_pages = set(
    df["landing_page"].dropna().unique()
)

unexpected_pages = observed_pages - expected_pages

missing_page_values = df["landing_page"].isna().sum()

print("\n========== VALIDATION RESULTS ==========")
print("Expected pages:", expected_pages)
print("Observed pages:", observed_pages)
print("Unexpected pages:", unexpected_pages)
print("Missing landing page values:", missing_page_values)

assert observed_pages == expected_pages
assert missing_page_values == 0

print("\nLanding page labels validated successfully.")


========== LANDING PAGE VALIDATION ==========

Observed landing page distribution:
landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64

========== VALIDATION RESULTS ==========
Expected pages: {'old_page', 'new_page'}
Observed pages: {'old_page', 'new_page'}
Unexpected pages: set()
Missing landing page values: 0

Landing page labels validated successfully.


In [9]:

# Step 03.21 — Validate Conversion Values

expected_conversion_values = {0, 1}

print("========== CONVERSION VALUE VALIDATION ==========")

# Inspect observed conversion values.
conversion_counts = df["converted"].value_counts(
    dropna=False
)

print("\nObserved conversion values:")
print(conversion_counts)

# Identify invalid values.
observed_conversion_values = set(
    df["converted"].dropna().unique()
)

unexpected_conversion_values = (
    observed_conversion_values - expected_conversion_values
)

missing_conversion_values = df["converted"].isna().sum()

print("\n========== VALIDATION RESULTS ==========")
print("Expected values:", expected_conversion_values)
print("Observed values:", observed_conversion_values)
print("Unexpected values:", unexpected_conversion_values)
print("Missing conversion values:", missing_conversion_values)

assert observed_conversion_values == expected_conversion_values
assert missing_conversion_values == 0

print("\nConversion values validated successfully.")


========== CONVERSION VALUE VALIDATION ==========

Observed conversion values:
converted
0    259241
1     35237
Name: count, dtype: int64

========== VALIDATION RESULTS ==========
Expected values: {0, 1}
Observed values: {np.int64(0), np.int64(1)}
Unexpected values: set()
Missing conversion values: 0

Conversion values validated successfully.


In [10]:

# Step 03.22 — Verify Missing Values and Exact Duplicates

print("========== MISSING VALUE VALIDATION ==========")

missing_by_column = df.isna().sum()

print("\nMissing values by column:")
print(missing_by_column)

total_missing = int(missing_by_column.sum())

print(f"\nTotal missing values: {total_missing:,}")

print("\n========== EXACT DUPLICATE VALIDATION ==========")

exact_duplicate_rows = int(df.duplicated().sum())

print(f"Exact duplicate rows: {exact_duplicate_rows:,}")

print("\n========== FINAL VALIDATION ==========")

assert total_missing == 0
assert exact_duplicate_rows == 0

print("Missing value validation: PASSED")
print("Exact duplicate validation: PASSED")
print("No records modified or removed.")


========== MISSING VALUE VALIDATION ==========

Missing values by column:
user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

Total missing values: 0

========== EXACT DUPLICATE VALIDATION ==========
Exact duplicate rows: 0

========== FINAL VALIDATION ==========
Missing value validation: PASSED
Exact duplicate validation: PASSED
No records modified or removed.


In [11]:

# Step 03.23 — Validate Repeated User IDs

print("========== REPEATED USER VALIDATION ==========")

# Count observations for each user.
user_observation_counts = df["user_id"].value_counts()

# Identify users appearing more than once.
repeated_user_counts = user_observation_counts[
    user_observation_counts > 1
]

total_unique_users = df["user_id"].nunique()

repeated_users = len(repeated_user_counts)

additional_rows = int(
    (repeated_user_counts - 1).sum()
)

single_observation_users = int(
    (user_observation_counts == 1).sum()
)

print(f"Total records: {len(df):,}")
print(f"Unique users: {total_unique_users:,}")
print(f"Users with one observation: {single_observation_users:,}")
print(f"Users with multiple observations: {repeated_users:,}")
print(f"Additional rows beyond first observation: {additional_rows:,}")

print("\n========== OBSERVATION FREQUENCY ==========")
print(
    user_observation_counts.value_counts().sort_index()
)

print("\n========== VALIDATION ==========")

assert total_unique_users == 290584
assert repeated_users == 3894
assert additional_rows == 3894

print("Repeated user audit validated successfully.")
print("No users or records have been removed.")


========== REPEATED USER VALIDATION ==========
Total records: 294,478
Unique users: 290,584
Users with one observation: 286,690
Users with multiple observations: 3,894
Additional rows beyond first observation: 3,894

========== OBSERVATION FREQUENCY ==========
count
1    286690
2      3894
Name: count, dtype: int64

========== VALIDATION ==========
Repeated user audit validated successfully.
No users or records have been removed.


In [12]:

# Step 03.24 — Validate Cross-Group User Assignments

print("========== CROSS-GROUP ASSIGNMENT AUDIT ==========")

# Count distinct experiment groups for each user.
user_group_counts = (
    df.groupby("user_id")["group"]
    .nunique()
)

# Identify users assigned to multiple groups.
cross_group_user_ids = user_group_counts[
    user_group_counts > 1
].index

cross_group_users = len(cross_group_user_ids)

# Identify repeated users who remain within one group.
within_group_repeated_users = (
    len(repeated_user_counts) - cross_group_users
)

print(f"Total unique users: {df['user_id'].nunique():,}")
print(f"Repeated users: {len(repeated_user_counts):,}")
print(f"Cross-group users: {cross_group_users:,}")
print(
    "Within-group repeated users:",
    f"{within_group_repeated_users:,}"
)

print("\n========== VALIDATION ==========")

assert cross_group_users == 1895
assert within_group_repeated_users == 1999

print("Cross-group assignment audit validated successfully.")
print("No users or records have been removed.")


========== CROSS-GROUP ASSIGNMENT AUDIT ==========
Total unique users: 290,584
Repeated users: 3,894
Cross-group users: 1,895
Within-group repeated users: 1,999

========== VALIDATION ==========
Cross-group assignment audit validated successfully.
No users or records have been removed.


In [13]:

# Step 03.25 — Validate Group–Landing Page Mismatches

print("========== GROUP–LANDING PAGE AUDIT ==========")

# Define the expected landing page for each group.
expected_page_mapping = {
    "control": "old_page",
    "treatment": "new_page"
}

# Identify mismatched observations without modifying df.
expected_page = df["group"].map(expected_page_mapping)

mismatch_mask = df["landing_page"] != expected_page

# Count mismatched records and affected users.
mismatched_records = int(mismatch_mask.sum())

mismatch_user_ids = df.loc[
    mismatch_mask, "user_id"
].unique()

mismatch_affected_users = len(mismatch_user_ids)

# Display the assignment/exposure cross-tabulation.
group_page_table = pd.crosstab(
    df["group"],
    df["landing_page"]
)

print("\nGroup–landing page distribution:")
print(group_page_table)

print("\n========== MISMATCH SUMMARY ==========")
print(f"Mismatched records: {mismatched_records:,}")
print(f"Mismatch-affected users: {mismatch_affected_users:,}")

print("\n========== VALIDATION ==========")

assert mismatched_records == 3893
assert mismatch_affected_users == 3893

print("Group–landing page mismatch audit validated.")
print("No records modified or removed.")


========== GROUP–LANDING PAGE AUDIT ==========

Group–landing page distribution:
landing_page  new_page  old_page
group                           
control           1928    145274
treatment       145311      1965

========== MISMATCH SUMMARY ==========
Mismatched records: 3,893
Mismatch-affected users: 3,893

========== VALIDATION ==========
Group–landing page mismatch audit validated.
No records modified or removed.


In [14]:

# Step 03.26 — Validate Overlap Between Data-Quality Issues

print("========== DATA-QUALITY ISSUE OVERLAP ==========")

# Identify users with repeated observations.
repeated_user_ids = set(
    repeated_user_counts.index
)

# Identify users appearing in both groups.
cross_group_ids = set(
    cross_group_user_ids
)

# Identify users affected by group–page mismatches.
mismatch_ids = set(
    mismatch_user_ids
)

# Combine all affected users without double-counting.
all_affected_users = (
    repeated_user_ids
    | cross_group_ids
    | mismatch_ids
)

# Identify users without these issues.
all_user_ids = set(df["user_id"].unique())

unaffected_users = all_user_ids - all_affected_users

print(f"Repeated users: {len(repeated_user_ids):,}")
print(f"Cross-group users: {len(cross_group_ids):,}")
print(f"Mismatch-affected users: {len(mismatch_ids):,}")

print("\n========== OVERLAP RESULTS ==========")

print(
    "Cross-group users with mismatches:",
    f"{len(cross_group_ids & mismatch_ids):,}"
)

print(
    "Repeated users with mismatches:",
    f"{len(repeated_user_ids & mismatch_ids):,}"
)

print(
    "Distinct users with at least one issue:",
    f"{len(all_affected_users):,}"
)

print(
    "Users without identified issues:",
    f"{len(unaffected_users):,}"
)

print("\n========== VALIDATION ==========")

assert len(all_affected_users) == 3894
assert len(unaffected_users) == 286690

assert (
    len(all_affected_users) + len(unaffected_users)
    == df["user_id"].nunique()
)

print("Data-quality overlap validation passed.")
print("No records modified or removed.")


========== DATA-QUALITY ISSUE OVERLAP ==========
Repeated users: 3,894
Cross-group users: 1,895
Mismatch-affected users: 3,893

========== OVERLAP RESULTS ==========
Cross-group users with mismatches: 1,895
Repeated users with mismatches: 3,893
Distinct users with at least one issue: 3,894
Users without identified issues: 286,690

========== VALIDATION ==========
Data-quality overlap validation passed.
No records modified or removed.


In [15]:

# Step 03.27 — Audit Chronological Observation Order

print("========== CHRONOLOGICAL REPEATED-USER AUDIT ==========")

# Use the previously validated timestamp conversion.
# This creates a temporary audit DataFrame.
df_chronological_audit = df.loc[
    df["user_id"].isin(repeated_user_ids),
    ["user_id", "timestamp", "group", "landing_page"]
].copy()

# Attach the corresponding parsed timestamps.
df_chronological_audit["parsed_timestamp"] = (
    parsed_timestamps.loc[df_chronological_audit.index]
)

# Count distinct timestamps for each repeated user.
timestamp_counts = (
    df_chronological_audit.groupby("user_id")[
        "parsed_timestamp"
    ].nunique()
)

# Check whether any repeated user has identical timestamps.
users_with_identical_timestamps = int(
    (timestamp_counts < 2).sum()
)

users_with_distinct_timestamps = int(
    (timestamp_counts == 2).sum()
)

print(f"Repeated users: {len(repeated_user_ids):,}")

print(
    "Users with distinct timestamps:",
    f"{users_with_distinct_timestamps:,}"
)

print(
    "Users with identical timestamps:",
    f"{users_with_identical_timestamps:,}"
)

print("\n========== VALIDATION ==========")

assert len(df_chronological_audit) == 7788
assert users_with_distinct_timestamps == 3894
assert users_with_identical_timestamps == 0

print("Chronological observation audit passed.")
print("No records selected, modified, or removed.")


========== CHRONOLOGICAL REPEATED-USER AUDIT ==========
Repeated users: 3,894
Users with distinct timestamps: 3,894
Users with identical timestamps: 0

========== VALIDATION ==========
Chronological observation audit passed.
No records selected, modified, or removed.


In [16]:

# Step 03.28 — Audit First and Second Group Assignments

print("========== CHRONOLOGICAL ASSIGNMENT AUDIT ==========")

# Sort repeated-user observations chronologically.
df_repeated_sorted = (
    df_chronological_audit
    .sort_values(
        ["user_id", "parsed_timestamp"],
        kind="stable"
    )
    .copy()
)

# Assign an observation number within each user.
df_repeated_sorted["observation_number"] = (
    df_repeated_sorted.groupby("user_id").cumcount() + 1
)

# Create a user-level table of first and second assignments.
assignment_sequence = df_repeated_sorted.pivot(
    index="user_id",
    columns="observation_number",
    values="group"
)

assignment_sequence.columns = [
    "first_group",
    "second_group"
]

# Display chronological group transitions.
transition_table = pd.crosstab(
    assignment_sequence["first_group"],
    assignment_sequence["second_group"]
)

print("\nFirst-to-second group transitions:")
print(transition_table)

# Count users whose group assignment changed.
changed_group_users = int(
    (
        assignment_sequence["first_group"]
        != assignment_sequence["second_group"]
    ).sum()
)

unchanged_group_users = int(
    (
        assignment_sequence["first_group"]
        == assignment_sequence["second_group"]
    ).sum()
)

print("\n========== ASSIGNMENT SUMMARY ==========")
print(f"Users changing groups: {changed_group_users:,}")
print(f"Users remaining in one group: {unchanged_group_users:,}")

print("\n========== VALIDATION ==========")

assert len(assignment_sequence) == 3894
assert changed_group_users == 1895
assert unchanged_group_users == 1999

print("Chronological assignment audit passed.")
print("No observations selected, modified, or removed.")


========== CHRONOLOGICAL ASSIGNMENT AUDIT ==========

First-to-second group transitions:
second_group  control  treatment
first_group                     
control          1007        994
treatment         901        992

========== ASSIGNMENT SUMMARY ==========
Users changing groups: 1,895
Users remaining in one group: 1,999

========== VALIDATION ==========
Chronological assignment audit passed.
No observations selected, modified, or removed.


In [17]:

# Step 03.29 — Audit Chronological Landing-Page Exposure

print("========== CHRONOLOGICAL PAGE EXPOSURE AUDIT ==========")

# Create a user-level table of first and second page exposures.
page_sequence = df_repeated_sorted.pivot(
    index="user_id",
    columns="observation_number",
    values="landing_page"
)

page_sequence.columns = [
    "first_page",
    "second_page"
]

# Display page exposure transitions.
page_transition_table = pd.crosstab(
    page_sequence["first_page"],
    page_sequence["second_page"]
)

print("\nFirst-to-second landing-page transitions:")
print(page_transition_table)

# Count users whose observed landing page changed.
changed_page_users = int(
    (
        page_sequence["first_page"]
        != page_sequence["second_page"]
    ).sum()
)

unchanged_page_users = int(
    (
        page_sequence["first_page"]
        == page_sequence["second_page"]
    ).sum()
)

print("\n========== PAGE EXPOSURE SUMMARY ==========")
print(f"Users changing pages: {changed_page_users:,}")
print(f"Users remaining on same page: {unchanged_page_users:,}")

print("\n========== VALIDATION ==========")

assert len(page_sequence) == 3894
assert changed_page_users == 1998
assert unchanged_page_users == 1896

print("Chronological page exposure audit passed.")
print("No records selected, modified, or removed.")


========== CHRONOLOGICAL PAGE EXPOSURE AUDIT ==========

First-to-second landing-page transitions:
second_page  new_page  old_page
first_page                     
new_page          922       972
old_page         1026       974

========== PAGE EXPOSURE SUMMARY ==========
Users changing pages: 1,998
Users remaining on same page: 1,896

========== VALIDATION ==========
Chronological page exposure audit passed.
No records selected, modified, or removed.


In [18]:

# Step 03.30 — Audit Mismatch Position Over Time

print("========== CHRONOLOGICAL MISMATCH AUDIT ==========")

# Identify mismatches in the chronological audit copy.
df_repeated_sorted["page_mismatch"] = (
    df_repeated_sorted["landing_page"]
    != df_repeated_sorted["group"].map(
        expected_page_mapping
    )
)

# Count mismatches in first and second observations.
mismatch_by_position = (
    df_repeated_sorted.groupby("observation_number")[
        "page_mismatch"
    ].sum()
)

print("\nMismatched records by observation position:")
print(mismatch_by_position)

# Create a user-level mismatch sequence.
mismatch_sequence = df_repeated_sorted.pivot(
    index="user_id",
    columns="observation_number",
    values="page_mismatch"
)

mismatch_sequence.columns = [
    "first_mismatch",
    "second_mismatch"
]

# Count mismatch patterns.
mismatch_patterns = (
    mismatch_sequence
    .value_counts()
    .rename("users")
)

print("\n========== MISMATCH PATTERNS ==========")
print(mismatch_patterns)

# Summarize users with mismatches.
users_with_any_mismatch = int(
    mismatch_sequence.any(axis=1).sum()
)

users_with_both_mismatches = int(
    mismatch_sequence.all(axis=1).sum()
)

print("\n========== SUMMARY ==========")
print(
    "Users with at least one mismatch:",
    f"{users_with_any_mismatch:,}"
)

print(
    "Users with mismatches in both observations:",
    f"{users_with_both_mismatches:,}"
)

print("\n========== VALIDATION ==========")

assert len(mismatch_sequence) == 3894
assert users_with_any_mismatch == 3893

assert int(mismatch_by_position.sum()) == 3893

print("Chronological mismatch audit passed.")
print("No original records modified or removed.")


========== CHRONOLOGICAL MISMATCH AUDIT ==========

Mismatched records by observation position:
observation_number
1    1949
2    1944
Name: page_mismatch, dtype: int64

========== MISMATCH PATTERNS ==========
first_mismatch  second_mismatch
True            False              1949
False           True               1944
                False                 1
Name: users, dtype: int64

========== SUMMARY ==========
Users with at least one mismatch: 3,893
Users with mismatches in both observations: 0

========== VALIDATION ==========
Chronological mismatch audit passed.
No original records modified or removed.


In [19]:

# Step 03.31 — Raw Sample-Ratio Mismatch (SRM) Check

from scipy.stats import chisquare

print("========== RAW SAMPLE-RATIO MISMATCH CHECK ==========")

# Approved expected allocation: 50% control, 50% treatment.
expected_allocation = [0.50, 0.50]

# Use raw record counts, as specified for the pre-cleaning check.
raw_group_counts = (
    df["group"]
    .value_counts()
    .reindex(["control", "treatment"])
)

observed_counts = raw_group_counts.to_numpy()

total_records = int(observed_counts.sum())

expected_counts = [
    total_records * proportion
    for proportion in expected_allocation
]

# Chi-square goodness-of-fit test.
chi_square_stat, srm_p_value = chisquare(
    f_obs=observed_counts,
    f_exp=expected_counts
)

# Approved SRM alarm threshold.
srm_threshold = 0.001

print(f"Control records: {observed_counts[0]:,}")
print(f"Treatment records: {observed_counts[1]:,}")
print(f"Total records: {total_records:,}")

print("\n========== SRM RESULTS ==========")
print(f"Chi-square statistic: {chi_square_stat:.6f}")
print(f"SRM p-value: {srm_p_value:.6f}")
print(f"SRM alarm threshold: {srm_threshold}")

if srm_p_value < srm_threshold:
    print("SRM STATUS: ALARM — INVESTIGATION REQUIRED")
else:
    print("SRM STATUS: NO ALARM")

print("\n========== VALIDATION ==========")

assert total_records == 294478
assert observed_counts.sum() == total_records
assert srm_threshold == 0.001

print("Raw SRM diagnostic completed successfully.")
print("No conversion outcomes examined.")
print("No records modified or removed.")


========== RAW SAMPLE-RATIO MISMATCH CHECK ==========
Control records: 147,202
Treatment records: 147,276
Total records: 294,478

========== SRM RESULTS ==========
Chi-square statistic: 0.018596
SRM p-value: 0.891532
SRM alarm threshold: 0.001
SRM STATUS: NO ALARM

========== VALIDATION ==========
Raw SRM diagnostic completed successfully.
No conversion outcomes examined.
No records modified or removed.


In [20]:

# Step 03.32 — Audit Earliest-Observation Eligibility

print("========== EARLIEST-OBSERVATION ELIGIBILITY AUDIT ==========")

# Create a temporary chronological copy of all observations.
df_earliest_audit = df.copy()

df_earliest_audit["parsed_timestamp"] = parsed_timestamps

# Sort by user and observation timestamp.
df_earliest_audit = df_earliest_audit.sort_values(
    ["user_id", "parsed_timestamp"],
    kind="stable"
)

# Identify each user's earliest recorded observation.
earliest_observations = (
    df_earliest_audit
    .drop_duplicates(subset=["user_id"], keep="first")
    .copy()
)

# Check whether group and page agree on the earliest record.
earliest_observations["page_matches_group"] = (
    earliest_observations["landing_page"]
    == earliest_observations["group"].map(
        expected_page_mapping
    )
)

matching_earliest = int(
    earliest_observations["page_matches_group"].sum()
)

mismatching_earliest = int(
    (~earliest_observations["page_matches_group"]).sum()
)

print(f"Unique users: {len(earliest_observations):,}")
print(f"Matching earliest observations: {matching_earliest:,}")
print(f"Mismatching earliest observations: {mismatching_earliest:,}")

print("\n========== EARLIEST GROUP COUNTS ==========")
print(earliest_observations["group"].value_counts())

print("\n========== VALIDATION ==========")

assert len(earliest_observations) == 290584
assert matching_earliest + mismatching_earliest == 290584
assert mismatching_earliest == 1949
assert df.equals(df_raw)

print("Earliest-observation eligibility audit passed.")
print("Original working data remains unchanged.")
print("No cleaning policy has been implemented.")


========== EARLIEST-OBSERVATION ELIGIBILITY AUDIT ==========
Unique users: 290,584
Matching earliest observations: 288,635
Mismatching earliest observations: 1,949

========== EARLIEST GROUP COUNTS ==========
group
control      145294
treatment    145290
Name: count, dtype: int64

========== VALIDATION ==========
Earliest-observation eligibility audit passed.
Original working data remains unchanged.
No cleaning policy has been implemented.


In [21]:

# Step 03.34A — Create Provisional User-Level Dataset

print("========== PROVISIONAL USER-LEVEL DATASET ==========")

# Copy the earliest-observation audit results.
# No changes are made to the original DataFrame.
df_clean = earliest_observations[
    ["user_id", "timestamp", "group", "landing_page", "converted"]
].copy(deep=True)

# Validate one observation per user.
total_users = len(df_clean)
unique_users = df_clean["user_id"].nunique()
duplicate_users = df_clean["user_id"].duplicated().sum()

print(f"Provisional dataset rows: {total_users:,}")
print(f"Unique users: {unique_users:,}")
print(f"Duplicate user IDs: {duplicate_users:,}")

print("\n========== GROUP COUNTS ==========")
print(df_clean["group"].value_counts())

print("\n========== VALIDATION ==========")

assert total_users == 290584
assert unique_users == 290584
assert duplicate_users == 0

# Confirm the original working dataset is unchanged.
assert df.equals(df_raw)

print("Provisional user-level dataset created successfully.")
print("Original working dataset remains unchanged.")
print("No conversion outcomes analyzed.")


========== PROVISIONAL USER-LEVEL DATASET ==========
Provisional dataset rows: 290,584
Unique users: 290,584
Duplicate user IDs: 0

========== GROUP COUNTS ==========
group
control      145294
treatment    145290
Name: count, dtype: int64

========== VALIDATION ==========
Provisional user-level dataset created successfully.
Original working dataset remains unchanged.
No conversion outcomes analyzed.


In [22]:

# Step 03.34B — Add Data-Quality Flags

print("========== DATA-QUALITY FLAGS ==========")

# Flag users who originally had multiple observations.
df_clean["was_repeated_user"] = (
    df_clean["user_id"].isin(repeated_user_ids)
)

# Flag users who appeared in both experiment groups.
df_clean["had_cross_group_assignment"] = (
    df_clean["user_id"].isin(cross_group_ids)
)

# Flag mismatches on the retained earliest observation.
df_clean["page_mismatch"] = (
    df_clean["landing_page"]
    != df_clean["group"].map(expected_page_mapping)
)

# Summarize each quality flag.
print("\nRepeated users:")
print(f"{df_clean['was_repeated_user'].sum():,}")

print("\nCross-group assignment users:")
print(f"{df_clean['had_cross_group_assignment'].sum():,}")

print("\nRetained observations with page mismatches:")
print(f"{df_clean['page_mismatch'].sum():,}")

print("\n========== VALIDATION ==========")

assert df_clean["was_repeated_user"].sum() == 3894
assert df_clean["had_cross_group_assignment"].sum() == 1895
assert df_clean["page_mismatch"].sum() == 1949

assert len(df_clean) == 290584
assert df_clean["user_id"].is_unique

# Cross-group users must also be repeated users.
assert (
    ~df_clean["had_cross_group_assignment"]
    | df_clean["was_repeated_user"]
).all()

# Verify the original working dataset remains unchanged.
assert df.equals(df_raw)

print("All data-quality flags validated successfully.")
print("No users excluded.")
print("Original working dataset remains unchanged.")
print("No conversion outcomes analyzed.")


========== DATA-QUALITY FLAGS ==========

Repeated users:
3,894

Cross-group assignment users:
1,895

Retained observations with page mismatches:
1,949

========== VALIDATION ==========
All data-quality flags validated successfully.
No users excluded.
Original working dataset remains unchanged.
No conversion outcomes analyzed.


In [23]:

# Step 03.35 — Final Structural Validation

print("========== FINAL STRUCTURAL VALIDATION ==========")

expected_columns = [
    "user_id",
    "timestamp",
    "group",
    "landing_page",
    "converted",
    "was_repeated_user",
    "had_cross_group_assignment",
    "page_mismatch"
]

# Validate column structure.
assert df_clean.columns.tolist() == expected_columns

# Validate user-level population.
assert len(df_clean) == 290584
assert df_clean["user_id"].nunique() == 290584
assert df_clean["user_id"].is_unique

# Validate missing values.
missing_values = int(df_clean.isna().sum().sum())

print(f"Rows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns)}")
print(f"Unique users: {df_clean['user_id'].nunique():,}")
print(f"Missing values: {missing_values:,}")

assert missing_values == 0

# Validate experiment groups and page labels.
assert set(df_clean["group"].unique()) == {
    "control", "treatment"
}

assert set(df_clean["landing_page"].unique()) == {
    "old_page", "new_page"
}

# Validate binary conversion values without
# calculating conversion rates or comparing groups.
assert set(df_clean["converted"].unique()) == {0, 1}

# Validate flag data types.
quality_flags = [
    "was_repeated_user",
    "had_cross_group_assignment",
    "page_mismatch"
]

for flag in quality_flags:
    assert pd.api.types.is_bool_dtype(df_clean[flag])

# Verify flags against their source definitions.
assert df_clean["was_repeated_user"].equals(
    df_clean["user_id"].isin(repeated_user_ids)
)

assert df_clean["had_cross_group_assignment"].equals(
    df_clean["user_id"].isin(cross_group_ids)
)

assert df_clean["page_mismatch"].equals(
    df_clean["landing_page"]
    != df_clean["group"].map(expected_page_mapping)
)

# Confirm raw working data remains unchanged.
assert df.equals(df_raw)

print("\n========== VALIDATION RESULTS ==========")
print("Column structure: PASSED")
print("User uniqueness: PASSED")
print("Missing values: PASSED")
print("Experiment categories: PASSED")
print("Binary outcome values: PASSED")
print("Data-quality flags: PASSED")
print("Original dataset integrity: PASSED")

print("\nAll structural validations passed successfully.")
print("No conversion outcomes analyzed.")


========== FINAL STRUCTURAL VALIDATION ==========
Rows: 290,584
Columns: 8
Unique users: 290,584
Missing values: 0

========== VALIDATION RESULTS ==========
Column structure: PASSED
User uniqueness: PASSED
Missing values: PASSED
Experiment categories: PASSED
Binary outcome values: PASSED
Data-quality flags: PASSED
Original dataset integrity: PASSED

All structural validations passed successfully.
No conversion outcomes analyzed.


In [24]:

# Step 03.36 — Post-Cleaning User-Level SRM Check

from scipy.stats import chisquare

print("========== USER-LEVEL SAMPLE-RATIO MISMATCH ==========")

# Approved allocation: 50% control, 50% treatment.
expected_allocation = [0.50, 0.50]

# Count unique users in each retained experiment group.
user_group_counts = (
    df_clean["group"]
    .value_counts()
    .reindex(["control", "treatment"], fill_value=0)
)

observed_counts = user_group_counts.to_numpy()

total_users = int(observed_counts.sum())

expected_counts = [
    total_users * proportion
    for proportion in expected_allocation
]

# Chi-square goodness-of-fit test.
chi_square_stat, srm_p_value = chisquare(
    f_obs=observed_counts,
    f_exp=expected_counts
)

# Approved SRM alarm threshold.
srm_threshold = 0.001

print(f"Control users: {observed_counts[0]:,}")
print(f"Treatment users: {observed_counts[1]:,}")
print(f"Total users: {total_users:,}")

print("\n========== EXPECTED COUNTS ==========")
print(f"Expected control: {expected_counts[0]:,.0f}")
print(f"Expected treatment: {expected_counts[1]:,.0f}")

print("\n========== SRM RESULTS ==========")
print(f"Chi-square statistic: {chi_square_stat:.6f}")
print(f"SRM p-value: {srm_p_value:.6f}")
print(f"Approved alarm threshold: {srm_threshold}")

if srm_p_value < srm_threshold:
    print("SRM STATUS: ALARM — INVESTIGATION REQUIRED")
else:
    print("SRM STATUS: NO ALARM")

print("\n========== VALIDATION ==========")

assert total_users == 290584
assert observed_counts[0] == 145294
assert observed_counts[1] == 145290
assert sum(expected_counts) == total_users
assert df_clean["user_id"].is_unique
assert df.equals(df_raw)

print("User-level SRM diagnostic completed successfully.")
print("No conversion outcomes examined.")
print("No original records modified.")


========== USER-LEVEL SAMPLE-RATIO MISMATCH ==========
Control users: 145,294
Treatment users: 145,290
Total users: 290,584

========== EXPECTED COUNTS ==========
Expected control: 145,292
Expected treatment: 145,292

========== SRM RESULTS ==========
Chi-square statistic: 0.000055
SRM p-value: 0.994079
Approved alarm threshold: 0.001
SRM STATUS: NO ALARM

========== VALIDATION ==========
User-level SRM diagnostic completed successfully.
No conversion outcomes examined.
No original records modified.


In [25]:

# Step 03.37A — Export Provisional User-Level Dataset

from pathlib import Path

print("========== PROVISIONAL DATASET EXPORT ==========")

# Locate the project root from the notebook working directory.
current_directory = Path.cwd().resolve()

if (
    (current_directory / "data").is_dir()
    and (current_directory / "notebooks").is_dir()
):
    project_root = current_directory

elif (
    (current_directory.parent / "data").is_dir()
    and (current_directory.parent / "notebooks").is_dir()
):
    project_root = current_directory.parent

else:
    raise FileNotFoundError(
        "Project root not found. Check the notebook working directory."
    )

# Save separately from the original raw dataset.
output_path = (
    project_root
    / "data"
    / "processed"
    / "ab_data_provisional_user_level.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Confirm the source dataset exists.
raw_path = project_root / "data" / "ab_data.csv"

assert raw_path.is_file(), (
    "Original raw CSV not found at the expected location."
)

assert output_path.resolve() != raw_path.resolve()

# Export the validated provisional dataset.
df_clean.to_csv(
    output_path,
    index=False
)

print(f"Project root: {project_root}")
print(f"Exported file: {output_path.relative_to(project_root)}")

print(f"Rows exported: {len(df_clean):,}")
print(f"Columns exported: {len(df_clean.columns)}")

print("\n========== VALIDATION ==========")

assert output_path.is_file()
assert output_path.stat().st_size > 0
assert df_clean["user_id"].is_unique
assert df.equals(df_raw)

print("Provisional dataset exported successfully.")
print("Original working dataset remains unchanged.")
print("No conversion outcomes analyzed.")


========== PROVISIONAL DATASET EXPORT ==========
Project root: C:\Users\kunal\OneDrive\Desktop\ab-test-experiment-analysis
Exported file: data\processed\ab_data_provisional_user_level.csv
Rows exported: 290,584
Columns exported: 8

========== VALIDATION ==========
Provisional dataset exported successfully.
Original working dataset remains unchanged.
No conversion outcomes analyzed.


In [30]:


# Step 03.37B — Corrected CSV Read-Back Validation

import pandas as pd
from pandas.testing import assert_frame_equal

print("========== EXPORTED DATASET READ-BACK ==========")

# Reload the exported CSV from disk.
df_exported = pd.read_csv(output_path)

print(f"Exported rows: {len(df_exported):,}")
print(f"Exported columns: {len(df_exported.columns)}")
print(f"Unique users: {df_exported['user_id'].nunique():,}")

# Validate dataset structure.
assert df_exported.shape == df_clean.shape

assert (
    df_exported.columns.tolist()
    == df_clean.columns.tolist()
)

assert df_exported["user_id"].is_unique

assert df_exported.isna().sum().sum() == 0

# Reset indexes only in temporary comparison copies.
# This does not modify either original DataFrame.
original_comparison = df_clean.reset_index(drop=True)
exported_comparison = df_exported.reset_index(drop=True)

# Compare every column, every value, and data types.
assert_frame_equal(
    original_comparison,
    exported_comparison,
    check_dtype=True,
    check_exact=True
)

# Confirm expected population and flags.
assert len(df_exported) == 290584

assert df_exported["was_repeated_user"].sum() == 3894

assert (
    df_exported["had_cross_group_assignment"].sum()
    == 1895
)

assert df_exported["page_mismatch"].sum() == 1949

# Confirm original working data remains unchanged.
assert df.equals(df_raw)

print("\n========== READ-BACK VALIDATION ==========")

print("Row count: PASSED")
print("Column structure: PASSED")
print("User uniqueness: PASSED")
print("Missing values: PASSED")
print("All column values: PASSED")
print("Data types: PASSED")
print("Data-quality flags: PASSED")
print("Original dataset integrity: PASSED")

print("\nExported dataset read-back validation PASSED.")
print("No conversion outcomes analyzed.")
print("No original records modified.")


========== EXPORTED DATASET READ-BACK ==========
Exported rows: 290,584
Exported columns: 8
Unique users: 290,584

========== READ-BACK VALIDATION ==========
Row count: PASSED
Column structure: PASSED
User uniqueness: PASSED
Missing values: PASSED
All column values: PASSED
Data types: PASSED
Data-quality flags: PASSED
Original dataset integrity: PASSED

Exported dataset read-back validation PASSED.
No conversion outcomes analyzed.
No original records modified.


In [29]:

# Step 03.37C — Diagnose User-ID Read-Back Failure

print("========== USER-ID COMPARISON DIAGNOSTIC ==========")

print("\nOriginal user ID dtype:")
print(df_clean["user_id"].dtype)

print("\nExported user ID dtype:")
print(df_exported["user_id"].dtype)

print("\nOriginal DataFrame index:")
print(df_clean.index[:5].tolist())

print("\nExported DataFrame index:")
print(df_exported.index[:5].tolist())

# Compare values position by position, ignoring indexes.
same_user_id_values = (
    df_clean["user_id"].to_numpy()
    == df_exported["user_id"].to_numpy()
).all()

print("\n========== COMPARISON RESULTS ==========")

print(f"Same number of rows: {len(df_clean) == len(df_exported)}")

print(f"User ID values identical: {same_user_id_values}")

print(
    "Indexes identical:",
    df_clean.index.equals(df_exported.index)
)

print(
    "Series.equals() result:",
    df_clean["user_id"].equals(df_exported["user_id"])
)

print("\nDiagnostic completed. No data modified.")


========== USER-ID COMPARISON DIAGNOSTIC ==========

Original user ID dtype:
int64

Exported user ID dtype:
int64

Original DataFrame index:
[63114, 103873, 205236, 247344, 242283]

Exported DataFrame index:
[0, 1, 2, 3, 4]

========== COMPARISON RESULTS ==========
Same number of rows: True
User ID values identical: True
Indexes identical: False
Series.equals() result: False

Diagnostic completed. No data modified.
